# Experiment 15: Combined Model-Wide Active + Inactive DBSCAN Tucker Compression (Kaggle Version)

**Target Model**: `google/gemma-3-1b-it` (All 26 Transformer Decoder Layers: `model.layers[0...25]`)  
**Target Submodules**: All 3 MLP Projections (`gate_proj`, `up_proj`, and `down_proj` -> 78 total matrices)  
**Evaluation Task**: GLUE MNLI Validation Set (`validation_matched`)  

### Core Architecture & Full-Matrix Factorization:
This benchmark synthesizes the complete functional compression architecture, factorizing **92.6% of every projection matrix** ($6,400$ out of $6,912$ coordinates) across all 78 submodules simultaneously:

1. **Active Core Compression (2,400 coordinates per matrix)**:
   - Clustered into 6 dense slices ($6 \times 400 \times 1152$) using independent activation DBSCAN.
   - Compressed with **Bespoke Adaptive Tucker** via HOSVD mode-unfolding singular value energy thresholding ($\tau_{\text{act}}$) + Adam GD on GPU.
2. **Inactive Subspace Compression (4,000 coordinates per matrix)**:
   - SVD 95% energy spectral denoising + 60% magnitude threshold sparsification.
   - Clustered into 10 uniform slices ($10 \times 400 \times 1152$) using fine-scale DBSCAN ($\epsilon_{\text{inact}} = \max(10^{-4}, 0.18 \times \operatorname{std}(v_{\text{inact}}))$).
   - Compressed with calibrated Inactive Tucker + Adam GD on GPU.
3. **Quarantined Superweights**:
   - Both active and inactive density noise (`-1`), extreme magnitudes, and top 1% variance outliers strictly preserved in uncompressed FP32.
4. **Massive Model-Wide Parameter Elimination**:
   - Cuts **400M to 500M parameters** across all 26 layers, testing downstream GLUE MNLI reasoning under full-model dual-subspace factorized operation!

In [ ]:
# Optional: Install required dependencies if not already present in your Kaggle environment
!pip install -q tensorly datasets scikit-learn


In [ ]:
# =====================================================================
# STEP 1: Environment Setup, Time Logging & Standard Imports
# =====================================================================
import os
import sys
import time
from pathlib import Path
import math
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from datasets import load_dataset
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
from IPython import get_ipython

# Set TensorLy PyTorch backend
tl.set_backend("pytorch")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Minimal Time Logging
GLOBAL_NOTEBOOK_START_TIME = time.time()
NOTEBOOK_TIMINGS = []
_current_cell_start = None

ip = get_ipython()
if ip is not None:
    def _pre_cell_hook(info):
        global _current_cell_start
        _current_cell_start = time.time()

    def _post_cell_hook(result):
        global _current_cell_start
        if _current_cell_start is not None:
            elapsed = time.time() - _current_cell_start
            cumulative = time.time() - GLOBAL_NOTEBOOK_START_TIME
            cell_id = result.execution_count or len(NOTEBOOK_TIMINGS) + 1

            timing_entry = {
                "cell_id": cell_id,
                "time": round(elapsed, 3),
                "cummulative_time": round(cumulative, 3),
            }
            NOTEBOOK_TIMINGS.append(timing_entry)

            print(f"time: {elapsed:.2f}s")
            print(f"cummulative_time: {cumulative:.2f}s")

    ip.events.register("pre_run_cell", _pre_cell_hook)
    ip.events.register("post_run_cell", _post_cell_hook)

print("Environment configured.")
print("TensorLy Backend:", tl.get_backend())
print("PyTorch Version: ", torch.__version__)
print("CUDA Available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:     ", torch.cuda.get_device_name(0))


In [ ]:
# =====================================================================
# STEP 2: Model & Dataset Loading (Pure Hugging Face - No Custom Module)
# =====================================================================
import huggingface_hub

MODEL_ID = "google/gemma-3-1b-it"
NUM_LAYERS = 26
NUM_EVAL_SAMPLES = 150

# Hugging Face Authentication for Gated Gemma-3 Model
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    huggingface_hub.login(token=hf_token)
    print("Authenticated with Hugging Face via token.")
else:
    print("No HF_TOKEN found in Kaggle Secrets or environment.")
    print("If you haven't yet, get your read token from https://huggingface.co/settings/tokens")
    print("and accept terms at https://huggingface.co/google/gemma-3-1b-it")
    huggingface_hub.login()

print(f"Loading model: {MODEL_ID} with device_map='auto' in FP32...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto",
    token=hf_token,
)

# Cache all 26 layers' pristine weights on CPU
W_orig_all = {}
for l in range(NUM_LAYERS):
    layer_mlp = model.model.layers[l].mlp
    W_orig_all[l] = {
        "gate_proj": layer_mlp.gate_proj.weight.data.clone().cpu(),
        "up_proj":   layer_mlp.up_proj.weight.data.clone().cpu(),
        "down_proj": layer_mlp.down_proj.weight.data.clone().cpu(),
    }

print(f"Cached all {NUM_LAYERS} layers pristine weights on CPU (78 projection matrices).")

# Load GLUE MNLI dataset
print(f"\nLoading GLUE MNLI dataset ({NUM_EVAL_SAMPLES} evaluation samples)...")
dataset = load_dataset("nyu-mll/glue", "mnli", split="validation_matched")
eval_data = dataset.select(range(NUM_EVAL_SAMPLES))

label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
label_tokens = ["entailment", "neutral", "contradiction"]
# Note: Gemma SentencePiece tokenization requires leading space for model answer generation
label_token_ids = [tokenizer.encode(" " + tok, add_special_tokens=False)[0] for tok in label_tokens]
print(f"Candidate label token IDs: {list(zip(label_tokens, label_token_ids))}")


In [ ]:
# =====================================================================
# STEP 3: Model-Wide Tri-Hook Profiling & Pristine Baseline Accuracy
# =====================================================================
# Memory-optimized streaming storage: pool sequence tokens immediately to prevent RAM bloat (<350 MB total)
layer_acts = {l: {"gate_proj": [], "up_proj": [], "down_proj": []} for l in range(NUM_LAYERS)}
hooks = []

for l in range(NUM_LAYERS):
    lmod = model.model.layers[l].mlp

    def make_gate_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["gate_proj"].append(
            out.detach().float().squeeze(0).abs().mean(dim=0).cpu()
        )

    def make_up_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["up_proj"].append(
            out.detach().float().squeeze(0).abs().mean(dim=0).cpu()
        )

    def make_down_hook(layer_idx):
        return lambda m, inp, out: layer_acts[layer_idx]["down_proj"].append(
            inp[0].detach().float().squeeze(0).abs().mean(dim=0).cpu()
        )

    hooks.append(lmod.act_fn.register_forward_hook(make_gate_hook(l)))
    hooks.append(lmod.up_proj.register_forward_hook(make_up_hook(l)))
    hooks.append(lmod.down_proj.register_forward_hook(make_down_hook(l)))

baseline_preds, ground_truths = [], []
model.eval()
print(f"Running baseline profiling pass across all {NUM_LAYERS} layers ({NUM_EVAL_SAMPLES} samples)...")
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Baseline Profiling"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs, logits_to_keep=1)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]
        pred_label = torch.argmax(candidate_logits).item()

        baseline_preds.append(pred_label)
        ground_truths.append(sample["label"])

for h in hooks:
    h.remove()

baseline_accuracy = accuracy_score(ground_truths, baseline_preds)
print(f"\nPristine Baseline Accuracy across all 26 layers: {baseline_accuracy * 100:.2f}%")

# Aggregate activation matrices per layer & submodule (Shape: [150, 6912])
acts_matrix_all = {l: {} for l in range(NUM_LAYERS)}
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        acts_matrix_all[l][sub_name] = torch.stack(layer_acts[l][sub_name], dim=0).numpy()

print(f"Successfully captured activation statistics for all 78 submodules ({acts_matrix_all[0]['gate_proj'].shape}).")


In [ ]:
# =====================================================================
# STEP 4: Dual-Subspace DBSCAN Pre-Clustering (Active + Inactive)
# =====================================================================
CHUNK_SIZE = 400
ACT_NUM_CHUNKS = 6    # 2,400 active coords
INACT_NUM_CHUNKS = 10 # 4,000 inactive coords -> 6,400 coords total (92.6% coverage!)

layer_dual_data = {}

def cluster_dual_submodule(acts_matrix, weight_tensor, is_col=False, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    v_all = np.mean(np.abs(acts_matrix), axis=0)
    var_all = np.var(acts_matrix, axis=0)
    
    # 1. Global Activity Partition
    sorted_all = np.argsort(v_all)
    inactive_pool = sorted_all[:-2400] # 4,512 coords
    active_pool = sorted_all[-2400:]   # 2,400 coords

    # 2. Notebook 02 Inactive Preprocessing: SVD 95% Denoising + 60% Sparsification on GPU
    W_sub_inact = (weight_tensor[:, inactive_pool].T if is_col else weight_tensor[inactive_pool, :]).to(device)
    U, S, Vh = torch.linalg.svd(W_sub_inact, full_matrices=False)
    cum_e = torch.cumsum(S**2, dim=0) / torch.sum(S**2)
    r95 = (cum_e >= 0.95).nonzero()[0].item() + 1
    W_denoised = U[:, :r95] @ torch.diag(S[:r95]) @ Vh[:r95, :]
    eps_val = torch.quantile(torch.abs(W_denoised), 0.60)
    W_sparse = W_denoised.clone()
    W_sparse[torch.abs(W_sparse) < eps_val] = 0.0
    
    W_clean = weight_tensor.clone().to(device)
    if is_col:
        W_clean[:, inactive_pool] = W_sparse.T
    else:
        W_clean[inactive_pool, :] = W_sparse
    W_clean = W_clean.cpu()
    
    # --- A. Active Subspace DBSCAN ---
    v_act = v_all[active_pool]
    eps_act = max(0.04, float(np.std(v_act) * 0.18))
    db_act = DBSCAN(eps=eps_act, min_samples=30, metric="euclidean")
    act_labels = db_act.fit_predict(v_act.reshape(-1, 1))
    
    act_mags = np.max(np.abs(acts_matrix[:, active_pool]), axis=0)
    act_vars = var_all[active_pool]
    act_super_mask = (act_labels == -1) | (act_mags >= np.quantile(act_mags, 0.99)) | (act_vars >= np.quantile(act_vars, 0.99))
    act_super_coords = active_pool[act_super_mask]
    act_clustered = active_pool[~act_super_mask]

    # --- B. Inactive Subspace DBSCAN ---
    v_inact = v_all[inactive_pool]
    eps_inact = max(1e-4, float(np.std(v_inact) * 0.18))
    db_inact = DBSCAN(eps=eps_inact, min_samples=30, metric="euclidean")
    inact_labels = db_inact.fit_predict(v_inact.reshape(-1, 1))
    
    inact_mags = np.max(np.abs(acts_matrix[:, inactive_pool]), axis=0)
    inact_vars = var_all[inactive_pool]
    inact_super_mask = (inact_labels == -1) | (inact_mags >= np.quantile(inact_mags, 0.99)) | (inact_vars >= np.quantile(inact_vars, 0.99))
    inact_super_coords = inactive_pool[inact_super_mask]
    inact_clustered = inactive_pool[~inact_super_mask]
    
    # Form active chunks
    act_chunks = []
    for lab in [l for l in np.unique(act_labels) if l != -1]:
        c_idx = np.where((act_labels == lab) & (~act_super_mask))[0]
        if len(c_idx) == 0: continue
        c_coords = active_pool[c_idx]
        sorted_c = c_coords[np.argsort(v_all[c_coords])]
        for ci in range(len(sorted_c) // CHUNK_SIZE):
            act_chunks.append(sorted_c[ci * CHUNK_SIZE : (ci + 1) * CHUNK_SIZE])
            if len(act_chunks) >= ACT_NUM_CHUNKS: break
        if len(act_chunks) >= ACT_NUM_CHUNKS: break
        
    # Guarantee exactly 6 active chunks
    if len(act_chunks) < ACT_NUM_CHUNKS:
        assigned = set(np.concatenate(act_chunks) if act_chunks else [])
        avail = [c for c in act_clustered if c not in assigned]
        needed_chunks = ACT_NUM_CHUNKS - len(act_chunks)
        for ci in range(needed_chunks):
            if len(avail) >= CHUNK_SIZE:
                act_chunks.append(np.array(avail[:CHUNK_SIZE]))
                avail = avail[CHUNK_SIZE:]
            else:
                break
        if len(act_chunks) < ACT_NUM_CHUNKS:
            assigned = set(np.concatenate(act_chunks) if act_chunks else [])
            avail_inact = [c for c in reversed(inact_clustered) if c not in assigned]
            needed_chunks = ACT_NUM_CHUNKS - len(act_chunks)
            for ci in range(needed_chunks):
                if len(avail_inact) >= CHUNK_SIZE:
                    act_chunks.append(np.array(avail_inact[:CHUNK_SIZE]))
                    avail_inact = avail_inact[CHUNK_SIZE:]
                
    # Form inactive chunks
    assigned_act = set(np.concatenate(act_chunks))
    inact_chunks = []
    for lab in [l for l in np.unique(inact_labels) if l != -1]:
        c_idx = np.where((inact_labels == lab) & (~inact_super_mask))[0]
        if len(c_idx) == 0: continue
        c_coords = [c for c in inactive_pool[c_idx] if c not in assigned_act]
        if len(c_coords) == 0: continue
        sorted_c = np.array(c_coords)[np.argsort(v_all[c_coords])]
        for ci in range(len(sorted_c) // CHUNK_SIZE):
            inact_chunks.append(sorted_c[ci * CHUNK_SIZE : (ci + 1) * CHUNK_SIZE])
            if len(inact_chunks) >= INACT_NUM_CHUNKS: break
        if len(inact_chunks) >= INACT_NUM_CHUNKS: break
        
    # Guarantee exactly 10 inactive chunks
    if len(inact_chunks) < INACT_NUM_CHUNKS:
        assigned_all = assigned_act.union(set(np.concatenate(inact_chunks) if inact_chunks else []))
        avail = [c for c in inact_clustered if c not in assigned_all]
        needed_inact = INACT_NUM_CHUNKS - len(inact_chunks)
        for ci in range(needed_inact):
            if len(avail) >= CHUNK_SIZE:
                inact_chunks.append(np.array(avail[:CHUNK_SIZE]))
                avail = avail[CHUNK_SIZE:]
                
    # --- C. Assemble 3D Tensors ---
    if is_col:
        T_act = torch.stack([weight_tensor[:, c].T.float().cpu() for c in act_chunks], dim=0)
        T_inact = torch.stack([W_clean[:, c].T.float().cpu() for c in inact_chunks], dim=0)
    else:
        T_act = torch.stack([weight_tensor[c, :].float().cpu() for c in act_chunks], dim=0)
        T_inact = torch.stack([W_clean[c, :].float().cpu() for c in inact_chunks], dim=0)
        
    all_super_coords = np.concatenate([act_super_coords, inact_super_coords])
    
    return {
        "T_act": T_act,
        "T_inact": T_inact,
        "act_chunks": act_chunks,
        "inact_chunks": inact_chunks,
        "super_coords": all_super_coords,
        "is_col": is_col,
    }

print("Running dual-subspace pre-clustering across all 26 layers on GPU...")
for l in tqdm(range(NUM_LAYERS), desc="Dual-Subspace Pre-Clustering"):
    layer_dual_data[l] = {}
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sub_mod = getattr(model.model.layers[l].mlp, sub_name)
        layer_dual_data[l][sub_name] = cluster_dual_submodule(
            acts_matrix_all[l][sub_name],
            W_orig_all[l][sub_name],
            is_col=(sub_name == "down_proj"),
            device=sub_mod.weight.device,
        )
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Dual-subspace pre-clustering completed for all 78 submodules on GPU.")


In [ ]:
# =====================================================================
# STEP 5: Define GPU Adaptive HOSVD Rank Selection & Tucker Adam GD
# =====================================================================
def compute_adaptive_ranks(T, tau, min_ranks=[2, 20, 50], device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    T_dev = T.to(device)
    s1 = torch.linalg.svdvals(tl.unfold(T_dev, 0))
    s2 = torch.linalg.svdvals(tl.unfold(T_dev, 1))
    s3 = torch.linalg.svdvals(tl.unfold(T_dev, 2))

    e1 = torch.cumsum(s1**2, dim=0) / torch.sum(s1**2)
    e2 = torch.cumsum(s2**2, dim=0) / torch.sum(s2**2)
    e3 = torch.cumsum(s3**2, dim=0) / torch.sum(s3**2)

    r1 = max(min_ranks[0], int((e1 >= tau).nonzero()[0].item()) + 1)
    r2 = max(min_ranks[1], int((e2 >= tau).nonzero()[0].item()) + 1)
    r3 = max(min_ranks[2], int((e3 >= tau).nonzero()[0].item()) + 1)

    r1 = min(r1, T.shape[0] - 1)
    r2 = min(r2, T.shape[1] - 1)
    r3 = min(r3, T.shape[2] - 1)

    return [r1, r2, r3]

def optimize_tucker_gd(T, ranks, num_steps=35, lr=1e-3, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    safe_ranks = [
        min(ranks[0], T.shape[0] - 1),
        min(ranks[1], T.shape[1] - 1),
        min(ranks[2], T.shape[2] - 1),
    ]
    T_target = T.to(device)
    core_init, factors_init = tucker(T_target, rank=safe_ranks, init='svd')
    core_param = torch.nn.Parameter(core_init.clone())
    factors_param = [torch.nn.Parameter(f.clone()) for f in factors_init]
    optimizer = torch.optim.Adam([core_param] + factors_param, lr=lr)

    for step in range(num_steps):
        optimizer.zero_grad()
        T_recon = tucker_to_tensor((core_param, factors_param))
        loss = torch.norm(T_target - T_recon) ** 2
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        T_recon_final = tucker_to_tensor((core_param, factors_param))
        final_err = (torch.norm(T_target - T_recon_final) / torch.norm(T_target)).item()

    return core_param.detach().cpu(), [f.detach().cpu() for f in factors_param], T_recon_final, final_err

combined_tiers = [
    {
        "name": "Combined Sweet Spot (Active: tau=0.65 | Inact: [5, 120, 350])",
        "act_tau": 0.65,
        "inact_ranks": [5, 120, 350],
    },
    {
        "name": "Combined High-Fidelity (Active: tau=0.70 | Inact: [6, 160, 500])",
        "act_tau": 0.70,
        "inact_ranks": [6, 160, 500],
    },
    {
        "name": "Combined Aggressive (Active: tau=0.60 | Inact: [4, 80, 200])",
        "act_tau": 0.60,
        "inact_ranks": [4, 80, 200],
    },
]

print("Defined GPU adaptive Tucker optimizer and 3 combined multi-tier configurations.")


In [ ]:
# =====================================================================
# STEP 6: Execute Combined Dual-Subspace Sweeps & GLUE MNLI Evaluation
# =====================================================================
combined_benchmarks = []

for tier in combined_tiers:
    tier_name = tier["name"]
    act_tau = tier["act_tau"]
    inact_ranks = tier["inact_ranks"]
    
    print(f"\n{'='*105}")
    print(f"Running Full Combined Dual-Subspace Evaluation: {tier_name}")
    print(f"{'='*105}")
    
    total_params_saved = 0
    gate_errs, up_errs, down_errs = [], [], []
    
    # 1. Decompose both Active and Inactive tensors and inject across all 26 layers
    for l in range(NUM_LAYERS):
        lmod = model.model.layers[l].mlp
        sdata_layer = layer_dual_data[l]
        
        for sub_name in ["gate_proj", "up_proj", "down_proj"]:
            sdata = sdata_layer[sub_name]
            T_act = sdata["T_act"]
            T_inact = sdata["T_inact"]
            mod_ref = getattr(lmod, sub_name)
            sub_dev = mod_ref.weight.device
            
            # --- Decompose Active Tensor with Bespoke Energy Ranks on GPU ---
            act_bespoke_ranks = compute_adaptive_ranks(T_act, tau=act_tau, device=sub_dev)
            cg_act, fg_act, T_act_recon, err_act = optimize_tucker_gd(
                T_act, ranks=act_bespoke_ranks, num_steps=35, lr=1e-3, device=sub_dev
            )
            
            # --- Decompose Inactive Tensor with Calibrated Inactive Ranks on GPU ---
            cg_inact, fg_inact, T_inact_recon, err_inact = optimize_tucker_gd(
                T_inact, ranks=inact_ranks, num_steps=35, lr=1e-3, device=sub_dev
            )
            
            # Parameter savings accounting
            p_saved_act = T_act.numel() - (cg_act.numel() + sum(f.numel() for f in fg_act))
            p_saved_inact = T_inact.numel() - (cg_inact.numel() + sum(f.numel() for f in fg_inact))
            total_params_saved += (p_saved_act + p_saved_inact)
            
            combined_err = (err_act * 2400 + err_inact * 4000) / 6400
            if sub_name == "gate_proj": gate_errs.append(combined_err)
            elif sub_name == "up_proj":  up_errs.append(combined_err)
            elif sub_name == "down_proj": down_errs.append(combined_err)
            
            # Live injection on GPU
            orig_w = W_orig_all[l][sub_name]
            mod_ref.weight.data = orig_w.clone().to(sub_dev)
            
            if sdata["is_col"]:
                for k, c in enumerate(sdata["act_chunks"]):
                    mod_ref.weight.data[:, c] = T_act_recon[k].T.to(device=sub_dev, dtype=mod_ref.weight.dtype)
                for k, c in enumerate(sdata["inact_chunks"]):
                    mod_ref.weight.data[:, c] = T_inact_recon[k].T.to(device=sub_dev, dtype=mod_ref.weight.dtype)
                if len(sdata["super_coords"]) > 0:
                    mod_ref.weight.data[:, sdata["super_coords"]] = orig_w[:, sdata["super_coords"]].to(sub_dev)
            else:
                for k, c in enumerate(sdata["act_chunks"]):
                    mod_ref.weight.data[c, :] = T_act_recon[k].to(device=sub_dev, dtype=mod_ref.weight.dtype)
                for k, c in enumerate(sdata["inact_chunks"]):
                    mod_ref.weight.data[c, :] = T_inact_recon[k].to(device=sub_dev, dtype=mod_ref.weight.dtype)
                if len(sdata["super_coords"]) > 0:
                    mod_ref.weight.data[sdata["super_coords"], :] = orig_w[sdata["super_coords"], :].to(sub_dev)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mean_gate = np.mean(gate_errs) * 100
    mean_up   = np.mean(up_errs) * 100
    mean_down = np.mean(down_errs) * 100
    
    print(f"Combined Factorization Complete across all 78 submodules.")
    print(f"  Mean Combined Recon Errors: gate={mean_gate:.1f}%, up={mean_up:.1f}%, down={mean_down:.1f}%")
    print(f"  Total Parameters Eliminated: {total_params_saved:,}")

    # 2. Evaluate downstream GLUE MNLI accuracy
    preds, gts = [], []
    model.eval()
    with torch.no_grad():
        for sample in tqdm(eval_data, desc=f"Evaluating {tier_name}"):
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            outputs = model(**inputs, logits_to_keep=1)

            next_token_logits = outputs.logits[0, -1, :]
            candidate_logits = next_token_logits[label_token_ids]
            pred_label = torch.argmax(candidate_logits).item()

            preds.append(pred_label)
            gts.append(sample["label"])

    acc = accuracy_score(gts, preds)
    delta = acc - baseline_accuracy

    print(f"\nResult for {tier_name}:")
    print(f"  Downstream Accuracy: {acc * 100:.2f}% (Δ vs Baseline: {delta * 100:+.2f}%)")
    print(f"  Total Params Cut:    {total_params_saved:,}")

    combined_benchmarks.append({
        "Variant": tier_name,
        "Act_Tau": act_tau,
        "Inact_Ranks": inact_ranks,
        "Mean_Gate_Err": round(mean_gate, 2),
        "Mean_Up_Err": round(mean_up, 2),
        "Mean_Down_Err": round(mean_down, 2),
        "Params_Eliminated": total_params_saved,
        "Accuracy": round(acc * 100, 2),
        "Delta": round(delta * 100, 2),
    })

# Restore pristine model weights across all 26 layers
for l in range(NUM_LAYERS):
    for sub_name in ["gate_proj", "up_proj", "down_proj"]:
        sub_mod = getattr(model.model.layers[l].mlp, sub_name)
        sub_mod.weight.data = W_orig_all[l][sub_name].clone().to(sub_mod.weight.device)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nRestored all 26 layers to pristine weights.")


In [ ]:
# =====================================================================
# STEP 7: Benchmark Summary & JSON Artifact Export
# =====================================================================
print(f"\n{'='*125}")
print(f"{'Variant':<55} | {'Gate Err':<9} | {'Up Err':<8} | {'Down Err':<9} | {'Params Cut':<12} | {'Accuracy':<9} | {'Delta':<8}")
print(f"{'='*125}")
print(f"{'Baseline (Uncompressed)':<55} | {'0.00%':<9} | {'0.00%':<8} | {'0.00%':<9} | {'0':<12} | {baseline_accuracy*100:>7.2f}% | {'+0.00%':<8}")

for b in combined_benchmarks:
    print(f"{b['Variant']:<55} | {b['Mean_Gate_Err']:>6.2f}%  | {b['Mean_Up_Err']:>5.2f}%  | {b['Mean_Down_Err']:>6.2f}%   | {b['Params_Eliminated']:<12,d} | {b['Accuracy']:>7.2f}% | {b['Delta']:>+6.2f}%")
print(f"{'='*125}")

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
results_file = artifacts_dir / "15_all_layers_combined_results.json"

payload = {
    "model_id": MODEL_ID,
    "num_layers": NUM_LAYERS,
    "baseline_accuracy": round(baseline_accuracy * 100, 2),
    "benchmarks": combined_benchmarks,
}

with open(results_file, "w") as f:
    json.dump(payload, f, indent=2)

print(f"\nSaved combined dual-subspace benchmark results to {results_file}")
